In [1]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
df = pd.read_csv('dataset_class.csv')

## עמודה N: vehicle_type
עמודה קטגוריאלית המציגה 6 סוגי רכבים המופיעים בדאטה.סוג הרכב ידוע ביום החתימה על ההסכם ולכן אין דליפת מידע. 

נבדוק את מספרי הנהגים בכל רכב, ערכים חסרים, ואת השפעת העמודה על משתנה המטרה

In [3]:
# בדיקת ערכים חסרים ורשימת סוגי הרכב עם מספר הנהגים בכל אחד

print("Missing:", df["vehicle_type"].isna().sum())
df["vehicle_type"].value_counts()

Missing: 0


vehicle_type
sedan         13916
small         12227
suv           10960
motorcycle     3072
pickup         2985
sports         1840
Name: count, dtype: int64

In [4]:
# השוואת עלות התביעות בין סוגי הרכב לפי שלושה מדדים
df.groupby("vehicle_type")["annual_claim_cost"].agg(
    avg_cost="mean",
    share_with_claim=lambda s: (s > 0).mean(),
    share_expensive=lambda s: (s > 10000).mean(),
    drivers="count",
).round(3).sort_values("avg_cost")

,avg_cost,share_with_claim,share_expensive,drivers
vehicle_type,,,,
small,1175.473,0.319,0.015,12227
sedan,2549.068,0.341,0.075,13916
motorcycle,2876.179,0.485,0.079,3072
suv,4051.931,0.338,0.143,10960
pickup,6483.999,0.435,0.214,2985
sports,8066.855,0.362,0.261,1840


### מסקנה לגבי vehicle_type
העמודה נקייה, עם פורמט אחיד וללא ערכים חסרים. יש הבדל גדול בעלות בין סוגי הרכב, פי 6.9 בין רכב קטן לרכב ספורט, העמודה תישאר.
סוגי הרכב שונים גם בתדירות התביעות וגם בחומרה. לאופנועים יש הכי הרבה תביעות (48.5%) אך מעט נהגים יקרים (7.9%). לרכבי ספורט יש אחוז תביעות רגיל (36%) אך הכי הרבה נהגים יקרים (26.1%).
לטנדרים יש גם הרבה תביעות וגם הרבה נהגים יקרים. ההשערה שלנו: טנדר הוא רכב עבודה שנוסעים בו הרבה, ולכן ייתכן שחלק מההשפעה שלו היא בעצם השפעה של סוג השימוש, נבדוק זאת בעמודה usage_type.

## עמודה O: vehicle_age
גיל הרכב בשנים. זו עמודה מספרית שידועה ביום החתימה ולכן אין כאן דליפת מידע.

נבדוק ערכים חסרים וערכים לא הגיוניים, את צורת ההתפלגות ואת הקשר לעלות התביעות.

In [9]:
# בדיקת ערכים חסרים וסטטיסטיקה תיאורית של גיל הרכב
print("Missing:", df["vehicle_age"].isna().sum())
df["vehicle_age"].describe().round(1)

Missing: 0


count    45000.0
mean         7.6
std          5.3
min          0.0
25%          4.0
50%          6.0
75%         10.0
max         25.0
Name: vehicle_age, dtype: float64

הממוצע גבוה מהחציון, נבדוק כמה רכבים יש בכל גיל והאם יש ערכים חריגים

In [12]:
df["vehicle_age"].value_counts().sort_index()

vehicle_age
0      723
1     2269
2     3489
3     4037
4     4176
5     4099
6     3862
7     3478
8     3017
9     2606
10    2241
11    1943
12    1652
13    1286
14    1158
15     875
16     786
17     606
18     498
19     414
20     312
21     268
22     249
23     185
24     163
25     608
Name: count, dtype: int64

In [13]:
# חלוקת הרכבים לחמש קבוצות שוות לפי גיל הרכב והשוואת עלות התביעות בין הקבוצות
age_group = pd.qcut(df["vehicle_age"], 5)
df.groupby(age_group, observed=True)["annual_claim_cost"].agg(
    avg_cost="mean",
    share_with_claim=lambda s: (s > 0).mean(),
    share_expensive=lambda s: (s > 10000).mean(),
    drivers="count",
).round(3)

,avg_cost,share_with_claim,share_expensive,drivers
vehicle_age,,,,
"(-0.001, 3.0]",4180.330,0.308,0.143,10518
"(3.0, 5.0]",3569.221,0.331,0.118,8275
"(5.0, 8.0]",3040.877,0.341,0.091,10357
"(8.0, 12.0]",2326.196,0.379,0.058,8442
"(12.0, 25.0]",1707.849,0.415,0.034,7408


In [6]:
# פונקציית הניקוי של עמודות הרכב והשימוש שנוסיף לה חוק אחרי כל עמודה שנבדוק
def clean_vehicle_cols(df):
    df = df.copy()

    # סוג רכב: הסרת רווחים והמרה לאותיות קטנות
    df["vehicle_type"] = df["vehicle_type"].str.strip().str.lower()

    return df

# בדיקה שהפונקציה רצה על כל הדאטה ועל שורה בודדת
print(clean_vehicle_cols(df).shape)
print(clean_vehicle_cols(df.iloc[[0]]).shape)

(45000, 30)
(1, 30)
